In [ ]:
import sys
sys.path.insert(0,'.')

In [ ]:
#| export
from google_auth import get_gdocs_service
from datetime import datetime, date # for type hints
from dateutil.parser import parse

In [ ]:
#| export
def str_to_date(content, year=None):
    if year is None:
        year = datetime.now().year
    return parse(content, default=datetime(year, 1, 1)).date()

In [ ]:
assert str_to_date('12/12')==date(datetime.now().year,12,12)
assert str_to_date('2026-12-12')==date(2026,12,12)
assert str_to_date('12/12',2024)==date(2024,12,12)

In [ ]:
#| export

def _get_paragraph_style(part):
    return (part.get('paragraph', {})
                .get('paragraphStyle', {})
                .get('namedStyleType'))
                

In [ ]:
#| export
def _is_heading_level(level:int,part): 
    return _get_paragraph_style(part)==f'HEADING_{level}'
    

In [ ]:
#| export
def _get_content(part):
    els = part.get('paragraph',{}).get('elements','')
    return ''.join([e.get('textRun',{}).get('content','') for e in els])
    

In [ ]:
#| export
def _get_parts_list(service,doc_id):
    doc = service.documents().get(documentId=doc_id).execute()
    return doc['body']['content']
    

In [ ]:
#| export
def _prepend_str(service, log_doc_id, pp_str, style='NORMAL_TEXT'):
    requests = [{
        'insertText': {
            'location': {'index': 1},
            'text': pp_str
            }
        },{
            'updateParagraphStyle': {
                'range': {'startIndex': 1, 'endIndex': len(pp_str)},
                'paragraphStyle': {'namedStyleType': style},
                'fields': 'namedStyleType'
            }
        }]
    service.documents().batchUpdate(documentId=log_doc_id, body={'requests': requests}).execute()
    

In [ ]:
#| export
LOGS = {
    "assembly": "1Jb1LEU_VVorhIkslgkGOUgRADny23Z1Q310GJSo6ZWg",
    "wcp": "1S4te4jvQGokB4cSbbInO9HuQUCEY6MHRontdIplXOBo",
    "bench": "1ezsnTsRJQE8NkVId5sOqXowKopygoov9xzIl35oKczk",
    "singulation": "1C6pDV0YWWwQra3sxPSRatdNV8DjPPdeBOvzxylLIUok",
    "sterilization": "1HUfc25DiOzTIVqnFQ0FEqNqEaxs7eTosMxVKJCov9KQ",
    "algorithm": "1nxCES9tzy2ZC0aFN-B08LO5lSs4YYVsgplED0v6GXxU",
    "firmware": "1KF8LlS0UbJCdVbcqp_7BEwScJQc4EHwfI6fxNCU-SKE",
    "quality": "1uXwbWHT3PjMFg6CJCIPs8ttQYTEdE4SYqm8wmH_AIvA",
    "clinicals": "1EXJ4kd531ewCqZpD-aPqKdlMfVmRmQ9LIRsppUhSLuk",
    "submission": "1Bbr8U9iDfYPrnoFKNeMgeMxYTN6uyeTTan5cvX_WmDI",
    "hardware": "1fvmtIe3AG3TEpkhKbONeEZO6ugOXnKKTjuWFHD6PPe8"
}


In [ ]:
#| export
def _log_to_dict_by_date(parts_list):
    log_dict = {}
    key = None
    year = None
    for part in parts_list:
        content = _get_content(part).strip()
        if _is_heading_level(2,part):
            year = int(content)
        if _is_heading_level(3,part) and content:
            key = str_to_date(content, year)
            log_dict[key] = ''
        else: 
            if key is not None: 
                log_dict[key] += content
    return log_dict
    

In [ ]:
#| export
def _get_log_entries_in_range(log_dict,start_date:date,end_date:date):
    return {str(k): v for k,v in log_dict.items() #str(date) for json
            if start_date <= k <= end_date}
            

In [ ]:
#| export
def _get_log_by_date(service,log_key, start_date:date, end_date:date):
    log_id = LOGS[log_key]
    parts_list = _get_parts_list(service,log_id)
    log_dict = _log_to_dict_by_date(parts_list)
    return _get_log_entries_in_range(log_dict, start_date, end_date)
    

In [ ]:
#| export
def read_logs(
        start_date_str:str, #first date in range
        end_date_str:str    #last date in range
    )->dict:
    "read all logs in the range"
    start_date,end_date = str_to_date(start_date_str),str_to_date(end_date_str)
    gdocs=get_gdocs_service()
    logs = {k: _get_log_by_date(gdocs,k,start_date,end_date) for k in LOGS}
    if not logs: print(f"No logs found for {start_date}-{end_date}")
    return logs

In [ ]:
#| export
def log_keys():
    return LOGS.keys()
    

In [ ]:
#| export
def write_log(key,digest,date):
    docs = get_gdocs_service()
    log_id=LOGS[key]
    _prepend_str(docs, log_id, digest + "\n", 'NORMAL_TEXT')
    _prepend_str(docs, log_id, f"{date}\n", 'HEADING_3')
    